# Cantera Equilibrium Accessibility Workflow v4

This notebook answers one core question:

> Given a starting inventory and a target-product list, which products are equilibrium-accessible under each modeled condition?

The default workflow is **single-product mode only**: each YAML contains the starting inventory species for one scenario plus one target product. This keeps the interpretation clean: each run asks whether that specific product is thermodynamically accessible when it is allowed as an equilibrium species.

Main outputs:

1. Raw equilibrium mole fractions and reconstructed moles
2. Target accessibility summary
3. Reactant depletion/source diagnostic
4. Combined accessibility bar charts
5. A final Markdown run summary

No yield analysis and no limiting-reagent normalization are used in v4.


## Cell 1 — User-editable settings

Edit this cell for a new project. Everything else in the notebook should run without manual changes.


In [1]:
# -----------------------------
# Main project settings
# -----------------------------
# This notebook ships configured for the bundled validation example
# (HCN + acetylene + ammonia in water -> adenine / cytosine, seeded from a
# public-CHNOSZ wide CSV). To run your own study, edit the INPUT FILES and
# CHNOSZ/CACHE BEHAVIOR sections below. See docs/USAGE.md for the full guide.
PROJECT_NAME = "cantera_equilibrium_workflow_v4"
PHASE_NAME = "aqueous"
MODEL_MODE = "single_product"
PRESSURE_PA = 1e5

# Thermodynamic fitting grid: keep this broad enough for NASA9 fitting.
THERMO_FIT_TEMPERATURE_GRID_C = [float(T) for T in range(0, 371, 10)]
T_FIT_SPLIT_K = 500.0

# Equilibrium diagnostic grid: only run the temperatures you want to inspect.
EQUILIBRIUM_TEMPERATURES_C = [20.0]

# Formation diagnostic: two-tier reporting.
#   X_eq >= SIGNIFICANT_X_THRESHOLD                          -> formation_call = 'significant'
#   FORMATION_X_THRESHOLD <= X_eq < SIGNIFICANT_X_THRESHOLD  -> formation_call = 'trace'
#   X_eq < FORMATION_X_THRESHOLD                             -> formation_call = 'below_threshold'
FORMATION_X_THRESHOLD = 1e-12
SIGNIFICANT_X_THRESHOLD = 1e-6
FORMATION_N_THRESHOLD_MOL = 0.0

# -----------------------------
# Input files (in inputs/)
# -----------------------------
SPECIES_FILENAME = "species_example.csv"
SCENARIO_FILENAME = "scenarios_example.yaml"

# -----------------------------
# CHNOSZ / cache behavior
# -----------------------------
# Two ways to populate the Gibbs-energy cache (see docs/USAGE.md "Seeding"):
#   A) Seed from an existing wide CSV (no pyCHNOSZ required). The bundled
#      example uses this so it runs out of the box.
#   B) Extract from CHNOSZ via pyCHNOSZ (set RUN_CHNOSZ_EXTRACTION = True).
#      Required for species CHNOSZ knows that are not already in the cache.
# These are not mutually exclusive: you can seed estimated species from CSV and
# extract CHNOSZ-known species in the same run.
SEED_CACHE_FROM_EXISTING_WIDE_CSV = True
SEED_WIDE_FILENAME = "example_validation_gibbs.csv"

# Map wide-CSV column names -> Cantera species names. Leave empty when the CSV
# column headers are already the Cantera names (as in the bundled example).
SEED_COLUMN_NAME_MAP = {}

RUN_CHNOSZ_EXTRACTION = False       # True when pyCHNOSZ is installed and species are missing from the cache
FORCE_REEXTRACT_CHNOSZ = False      # True only to rebuild the cache from scratch
EXCEED_TTR = True                   # passed to pyCHNOSZ.subcrt

# -----------------------------
# NASA9 fitting and YAML behavior
# -----------------------------
RUN_FIT_DIAGNOSTIC_PLOTS = True
FORCE_REFIT_THERMO = True
FORCE_REGENERATE_YAML = True
VALIDATE_YAML_WITH_CANTERA = True

# -----------------------------
# Cantera run behavior
# -----------------------------
RUN_CANTERA_EQUILIBRIUM = True
CANTERA_SOLVER = "vcs"
CANTERA_MAX_STEPS = 100_000
TARGET_PRODUCTS = None              # None -> use scenario target_products, else all role=product/target species

# -----------------------------
# Accessibility plot behavior
# -----------------------------
# Use "auto" for most cases. Use "fixed" + PLOT_X_AXIS_MAX to force the x-axis
# to end at a value like 1e4 or 1e8.
PLOT_X_AXIS_MAX_MODE = "auto"    # "auto" or "fixed"
PLOT_X_AXIS_MAX = None           # example fixed values: 1e4, 1e8
PLOT_X_AXIS_PADDING_FACTOR = 30.0
PLOT_FIGURE_WIDTH = 8.8
PLOT_SAVE_PNG = False

print("User settings loaded.")
print(f"  Project                 : {PROJECT_NAME}")
print(f"  Species file            : inputs/{SPECIES_FILENAME}")
print(f"  Scenario file           : inputs/{SCENARIO_FILENAME}")
print(f"  Equilibrium temperatures: {EQUILIBRIUM_TEMPERATURES_C} C")
print(f"  Formation threshold     : {FORMATION_X_THRESHOLD:.0e}")
print(f"  Plot x-axis mode        : {PLOT_X_AXIS_MAX_MODE}")


User settings loaded.
  Project                 : cantera_equilibrium_workflow_v4
  Species file            : inputs/species_example.csv
  Scenario file           : inputs/scenarios_example.yaml
  Equilibrium temperatures: [20.0] C
  Formation threshold     : 1e-12
  Plot x-axis mode        : auto


## Cell 2 — Project setup and imports

This cell sets paths, imports reusable modules, and creates output directories.


In [2]:
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Robust project-root detection: works when running from notebooks/ or project root.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config_io import (
    get_project_paths,
    load_species_metadata,
    load_scenarios,
    list_target_products,
    base_species_for_scenario,
)
from chnosz_cache import (
    load_cache,
    seed_cache_from_wide_csv,
    find_missing_rows,
    update_cache_with_missing,
    make_gibbs_wide,
)
from thermo_fit import fit_all_species
from cantera_yaml import (
    generate_single_product_yamls_for_scenarios,
    validate_yaml_with_cantera,
)
from equilibrium_runner import run_equilibrium_manifest, make_raw_wide
from mole_balance import add_equilibrium_moles
from result_summary import summarize_target_formation, target_formation_pivot
from diagnostics import (
    make_equilibrium_inspection_table,
    summarize_reactant_depletion,
    write_run_summary,
)
from plotting import plot_combined_accessibility_barchart

PATHS = get_project_paths(PROJECT_ROOT)
SPECIES_FILE = PATHS.inputs / SPECIES_FILENAME
SCENARIO_FILE = PATHS.inputs / SCENARIO_FILENAME
SEED_WIDE_CSV = PATHS.inputs / SEED_WIDE_FILENAME

print(f"Project root : {PATHS.root}")
print(f"src/         : {PATHS.src}")
print(f"inputs/      : {PATHS.inputs}")
print(f"data/results/: {PATHS.data_results}")
print()
print("All modules imported and output directories created.")


Project root : /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow
src/         : /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/src
inputs/      : /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/inputs
data/results/: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results

All modules imported and output directories created.


## Cell 3 — Load species and scenarios

The species CSV defines molecule names, formulas, CHNOSZ names, molar volumes, roles, and optionally product classes.

The scenario YAML defines fixed initial inventories. Molecules such as `NH3(aq)` are treated the same as any other molecule: include them in `initial_moles` only when they belong in that scenario.

**Important:** Cantera can form any species present in the YAML. Therefore, this workflow generates **scenario-specific single-product YAMLs** so absent species are not accidentally allowed to form.


In [3]:
species_df = load_species_metadata(SPECIES_FILE)
scenarios = load_scenarios(SCENARIO_FILE)

if TARGET_PRODUCTS is None:
    target_products = list_target_products(species_df)
else:
    target_products = list(TARGET_PRODUCTS)

# ── Species breakdown ────────────────────────────────────────────────────
print("=== Species metadata ===")
_display_cols = [c for c in ["species_key", "cantera_name", "formula", "state", "role", "product_class"] if c in species_df.columns]
display(species_df[_display_cols].fillna(""))

role_counts = species_df["role"].value_counts()
print()
print(f"  {len(species_df)} species total:")
for role, n in role_counts.items():
    print(f"    {role:12s}: {n}")

print()
print(f"  Target products ({len(target_products)}):")
for p in target_products:
    print(f"    • {p}")

# ── Scenario inventory ───────────────────────────────────────────────────
print()
print("=== Scenario initial inventories ===")
for scenario_id, cfg in scenarios["scenarios"].items():
    base = base_species_for_scenario(cfg)
    desc = cfg.get("description", "")
    print(f"  Scenario: {scenario_id}")
    if desc:
        print(f"    Description: {desc}")
    print(f"    Base/allowed species: {base}")
    print(f"    Initial moles:")
    for sp, mol in cfg["initial_moles"].items():
        print(f"      {sp:<24s} {float(mol):.8g} mol")
    if cfg.get("target_products"):
        print(f"    Scenario-specific target products: {cfg['target_products']}")
    print()

n_yaml_cases = 0
for _, cfg in scenarios["scenarios"].items():
    n_yaml_cases += len(cfg.get("target_products") or target_products)

print("NOTE: Each single-product YAML contains the starting inventory species + one target product.")
print("Cantera cannot form species that are absent from the YAML, even if thermodynamically favourable.")
print()
print(f"Estimated run count: {n_yaml_cases} YAML case(s) × {len(EQUILIBRIUM_TEMPERATURES_C)} temperature(s) = {n_yaml_cases * len(EQUILIBRIUM_TEMPERATURES_C)} equilibrium run(s)")


=== Species metadata ===


,species_key,cantera_name,formula,state,role,product_class
0,h2o,H2O(l),H2O,liq,solvent,
1,hcn,HCN(aq),CHN,aq,reactant,
2,c2h2,C2H2(aq),C2H2,aq,reactant,
3,nh3,NH3(aq),H3N,aq,reactant,
4,adenine,Adenine(aq),C5H5N5,aq,product,nucleobase
5,cytosine,Cytosine(aq),C4H5N3O,aq,product,nucleobase



  6 species total:
    reactant    : 3
    product     : 2
    solvent     : 1

  Target products (2):
    • Adenine(aq)
    • Cytosine(aq)

=== Scenario initial inventories ===
  Scenario: validation
    Description: Validation run: HCN + C2H2 + NH3 in water, 20 C
    Base/allowed species: ['H2O(l)', 'HCN(aq)', 'C2H2(aq)', 'NH3(aq)']
    Initial moles:
      H2O(l)                   1 mol
      HCN(aq)                  0.02 mol
      C2H2(aq)                 0.042 mol
      NH3(aq)                  0.02 mol
    Scenario-specific target products: ['Adenine(aq)', 'Cytosine(aq)']

NOTE: Each single-product YAML contains the starting inventory species + one target product.
Cantera cannot form species that are absent from the YAML, even if thermodynamically favourable.

Estimated run count: 2 YAML case(s) × 1 temperature(s) = 2 equilibrium run(s)


## Cell 4 — CHNOSZ cache check and extraction

This cell updates `data/raw/chnosz_gibbs_cache.csv`.

Behavior:

- If a requested species-temperature value already exists in the cache, it is reused.
- If a species-temperature value is missing, the notebook either extracts it from CHNOSZ or stops and shows what is missing.
- This avoids rerunning CHNOSZ unnecessarily when only some future molecules are new.


In [4]:
CACHE_CSV = PATHS.data_raw / "chnosz_gibbs_cache.csv"

if FORCE_REEXTRACT_CHNOSZ and CACHE_CSV.exists():
    CACHE_CSV.unlink()
    print("Removed existing cache because FORCE_REEXTRACT_CHNOSZ=True")

if SEED_CACHE_FROM_EXISTING_WIDE_CSV:
    cache_df = seed_cache_from_wide_csv(
        wide_csv=SEED_WIDE_CSV,
        species_df=species_df,
        cache_path=CACHE_CSV,
        name_map=SEED_COLUMN_NAME_MAP,
        source="seeded validation CSV",
    )
    print(f"Seeded/updated cache from {SEED_WIDE_CSV.name}: {CACHE_CSV}")
else:
    cache_df = load_cache(CACHE_CSV)

missing_df = find_missing_rows(cache_df, species_df, THERMO_FIT_TEMPERATURE_GRID_C)

# ── Cache status ─────────────────────────────────────────────────────────
print("=== CHNOSZ Gibbs-energy cache ===")
if cache_df.empty:
    print("  Cache is empty — all values will be extracted from CHNOSZ.")
else:
    cached_species = cache_df["cantera_name"].nunique()
    t_min = cache_df["T_C"].min()
    t_max = cache_df["T_C"].max()
    print(f"  Cached species    : {cached_species}")
    print(f"  Total rows        : {len(cache_df)}")
    print(f"  Temperature range : {t_min:.0f} – {t_max:.0f} °C")

if missing_df.empty:
    print()
    print("  ✓ Cache is complete — no CHNOSZ extraction needed for this run.")
else:
    print()
    print(f"  ✗ {len(missing_df)} species-temperature rows are missing from the cache.")
    print("    Missing rows (CHNOSZ will be queried for these):")
    display(missing_df.groupby("cantera_name")["T_C"].count().rename("missing_T_points"))
    if RUN_CHNOSZ_EXTRACTION:
        cache_df = update_cache_with_missing(
            species_df=species_df,
            temperatures_C=THERMO_FIT_TEMPERATURE_GRID_C,
            cache_path=CACHE_CSV,
            force_reextract=False,
            exceed_Ttr=EXCEED_TTR,
        )
        print("  Cache updated from CHNOSZ.")
    else:
        raise RuntimeError(
            "Missing species-temperature rows. Either set RUN_CHNOSZ_EXTRACTION=True "
            "in an environment with pyCHNOSZ installed, or seed/populate the cache manually."
        )

cache_df = load_cache(CACHE_CSV)
print()
print(f"Cache ready: {len(cache_df)} rows, {cache_df['cantera_name'].nunique()} species.")
print()
print("NOTE: The cache stores G(T) in J/mol from CHNOSZ. Once a species-temperature")
print("row is cached it is never re-queried, so you can add new species without")
print("rerunning old ones. Set FORCE_REEXTRACT_CHNOSZ=True only to start fresh.")


Seeded/updated cache from example_validation_gibbs.csv: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/raw/chnosz_gibbs_cache.csv
=== CHNOSZ Gibbs-energy cache ===
  Cached species    : 6
  Total rows        : 228
  Temperature range : 0 – 370 °C

  ✓ Cache is complete — no CHNOSZ extraction needed for this run.

Cache ready: 228 rows, 6 species.

NOTE: The cache stores G(T) in J/mol from CHNOSZ. Once a species-temperature
row is cached it is never re-queried, so you can add new species without
rerunning old ones. Set FORCE_REEXTRACT_CHNOSZ=True only to start fresh.


## Cell 5 — Create the fitting table

This converts the long CHNOSZ cache into a wide table for NASA9 fitting:

```text
T_K, species_1, species_2, species_3, ...
```

Output:

- `data/processed/gibbs_for_fitting.csv`


In [5]:
GIBBS_WIDE_CSV = PATHS.data_processed / "gibbs_for_fitting.csv"

gibbs_wide = make_gibbs_wide(
    cache_df=cache_df,
    species_df=species_df,
    temperatures_C=THERMO_FIT_TEMPERATURE_GRID_C,
    output_path=GIBBS_WIDE_CSV,
    use_column="cantera_name",
)

# ── Wide table diagnostics ───────────────────────────────────────────────
species_cols = [c for c in gibbs_wide.columns if c != "T_K"]
n_nan = gibbs_wide[species_cols].isna().sum()
has_nan = n_nan[n_nan > 0]

print("=== Gibbs fitting table ===")
print(f"  Shape            : {gibbs_wide.shape[0]} temperatures × {len(species_cols)} species")
print(f"  T_K range        : {gibbs_wide['T_K'].min():.2f} – {gibbs_wide['T_K'].max():.2f} K")
print(f"  Species columns  : {species_cols}")
if has_nan.empty:
    print("  ✓ No missing G values — table is complete.")
else:
    print(f"  ✗ Missing G values found (will cause fit to fail):")
    print(has_nan)

print()
print("NOTE: This wide table is an intermediate — not a final result.")
print("Each column is a species G(T) curve that will be fitted by two NASA9")
print("polynomial segments (low T and high T, split at T_FIT_SPLIT_K).")
display(gibbs_wide.head())
print(f"Saved: {GIBBS_WIDE_CSV}")


=== Gibbs fitting table ===
  Shape            : 38 temperatures × 6 species
  T_K range        : 273.16 – 643.15 K
  Species columns  : ['Adenine(aq)', 'C2H2(aq)', 'Cytosine(aq)', 'H2O(l)', 'HCN(aq)', 'NH3(aq)']
  ✓ No missing G values — table is complete.

NOTE: This wide table is an intermediate — not a final result.
Each column is a species G(T) curve that will be fitted by two NASA9
polynomial segments (low T and high T, split at T_FIT_SPLIT_K).


cantera_name,T_K,Adenine(aq),C2H2(aq),Cytosine(aq),H2O(l),HCN(aq),NH3(aq)
0,273.16,318198.596275,219981.947351,-30358.283864,-235515.256860,122578.697149,-24091.333152
1,283.15,316108.479709,218901.188164,-32261.400220,-236161.448309,121464.363397,-25118.002083
2,293.15,313945.951399,217725.880436,-34202.130543,-236834.941221,120278.543876,-26170.584237
3,303.15,311711.255553,216471.043977,-36195.758810,-237534.147744,119031.969356,-27248.805236
4,313.15,309403.813813,215145.633461,-38251.556472,-238258.217333,117730.970194,-28352.796625


Saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/gibbs_for_fitting.csv


## Cell 6 — Fit NASA9 coefficients

This fits two NASA9 coefficient ranges per species, split at `T_FIT_SPLIT_K`.

Outputs:

- `data/processed/nasa9_coefficients.csv`
- `data/processed/nasa9_fit_diagnostics.csv`
- fit/residual figures in `figures/fit_diagnostics/`

Inspect the diagnostics. Large residuals may mean the temperature range, split, or underlying thermodynamic data need attention.


In [6]:
COEFFS_CSV = PATHS.data_processed / "nasa9_coefficients.csv"
FIT_DIAGNOSTICS_CSV = PATHS.data_processed / "nasa9_fit_diagnostics.csv"

# Clear stale fit-diagnostic figures so the folder matches this run.
if RUN_FIT_DIAGNOSTIC_PLOTS:
    for _f in PATHS.figures_fit.glob("*.png"):
        _f.unlink()

if FORCE_REFIT_THERMO or not COEFFS_CSV.exists():
    coeff_df, fit_diag_df = fit_all_species(
        gibbs_wide_csv=GIBBS_WIDE_CSV,
        coefficients_csv=COEFFS_CSV,
        diagnostics_csv=FIT_DIAGNOSTICS_CSV,
        figures_dir=PATHS.figures_fit,
        T_split=T_FIT_SPLIT_K,
        make_plots=RUN_FIT_DIAGNOSTIC_PLOTS,
    )
else:
    coeff_df = pd.read_csv(COEFFS_CSV)
    fit_diag_df = pd.read_csv(FIT_DIAGNOSTICS_CSV)

# ── Fit quality diagnostics ──────────────────────────────────────────────
print("=== NASA9 fit quality ===")
print()
print("Each species G(T) is represented by two 9-coefficient polynomial segments")
print(f"split at {T_FIT_SPLIT_K:.0f} K. The residuals below are (fit − observed) in J/mol.")
print()

# Annotate with quality tier
def _tier(max_abs):
    if max_abs < 3000:   return "looks good"
    return "⚠ inspect"

fit_diag_df["quality"] = fit_diag_df["max_abs_residual_J_mol"].apply(_tier)
display(fit_diag_df[["cantera_name", "rmse_J_mol", "max_abs_residual_J_mol", "mean_abs_residual_J_mol", "quality"]]
        .rename(columns={"cantera_name": "species",
                          "rmse_J_mol": "RMSE (J/mol)",
                          "max_abs_residual_J_mol": "max |resid| (J/mol)",
                          "mean_abs_residual_J_mol": "mean |resid| (J/mol)"}))

print()
print("NOTE: These residuals look large in isolation but are tiny relative to G.")
print()
print("For example, 200 J/mol on a species with G ≈ -560,000 J/mol is only 0.04%.")
print()
print("The fit plots in figures/fit_diagnostics/ show observed vs fitted G(T) curves.")
print()
print(f"Saved: {COEFFS_CSV}")


=== NASA9 fit quality ===

Each species G(T) is represented by two 9-coefficient polynomial segments
split at 500 K. The residuals below are (fit − observed) in J/mol.



,species,RMSE (J/mol),max |resid| (J/mol),mean |resid| (J/mol),quality
0,Adenine(aq),243.587090,1070.508762,115.934839,looks good
1,C2H2(aq),119.796610,496.362434,60.961991,looks good
2,Cytosine(aq),28.565900,131.849949,12.631828,looks good
3,H2O(l),0.066507,0.285553,0.041750,looks good
4,HCN(aq),27.365041,116.213445,13.693680,looks good
5,NH3(aq),13.482135,55.351887,7.047043,looks good



NOTE: These residuals look large in isolation but are tiny relative to G.

For example, 200 J/mol on a species with G ≈ -560,000 J/mol is only 0.04%.

The fit plots in figures/fit_diagnostics/ show observed vs fitted G(T) curves.

Saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/nasa9_coefficients.csv


## Cell 7 — Generate single-product Cantera YAML files

This generates one YAML per scenario and target product.

Why scenario-specific? Because **species absence matters** in Cantera. If a molecule is present in a YAML, Cantera can potentially form it even if the initial amount is zero. Scenario-specific YAMLs keep each scenario chemically explicit.

Outputs:

- `models/single_product/<scenario>/<target>.yaml`
- `models/single_product/single_product_manifest.csv`


In [7]:
if FORCE_REGENERATE_YAML:
    # Clear old YAMLs/manifests so the models folder matches this run exactly.
    if PATHS.models_single.exists():
        shutil.rmtree(PATHS.models_single)
    PATHS.models_single.mkdir(parents=True, exist_ok=True)

    single_manifest = generate_single_product_yamls_for_scenarios(
        species_df=species_df,
        coeffs_csv=COEFFS_CSV,
        scenarios=scenarios,
        output_dir=PATHS.models_single,
        target_products=target_products,
        phase_name=PHASE_NAME,
    )
else:
    single_manifest = pd.read_csv(PATHS.models_single / "single_product_manifest.csv")

# ── YAML manifest diagnostics ────────────────────────────────────────────
print("=== Single-product YAML manifest ===")
print()
print("One YAML is generated per (scenario × target product).")
print("Each YAML contains: the base/solvent species for that scenario + one target product.")
print()
print("Why one YAML per product? Cantera can form ANY species present in a YAML,")
print("even if its initial amount is zero. Keeping products isolated means each")
print("run answers: 'is THIS product thermodynamically accessible under this starting inventory?'")
print()
display(single_manifest[["scenario", "target_product", "yaml_file", "allowed_base_species"]])
print()
print(f"  {len(single_manifest)} YAML cases generated across {single_manifest['scenario'].nunique()} scenario(s).")

# Check all files actually exist
missing_yamls = [row["yaml_file"] for _, row in single_manifest.iterrows()
                 if not (PATHS.root / row["yaml_path"]).exists()]
if missing_yamls:
    print(f"  ✗ {len(missing_yamls)} YAML file(s) missing on disk: {missing_yamls}")
else:
    print("  ✓ All YAML files confirmed on disk.")


=== Single-product YAML manifest ===

One YAML is generated per (scenario × target product).
Each YAML contains: the base/solvent species for that scenario + one target product.

Why one YAML per product? Cantera can form ANY species present in a YAML,
even if its initial amount is zero. Keeping products isolated means each
run answers: 'is THIS product thermodynamically accessible under this starting inventory?'



,scenario,target_product,yaml_file,allowed_base_species
0,validation,Adenine(aq),Adenine.yaml,H2O(l);HCN(aq);C2H2(aq);NH3(aq)
1,validation,Cytosine(aq),Cytosine.yaml,H2O(l);HCN(aq);C2H2(aq);NH3(aq)



  2 YAML cases generated across 1 scenario(s).
  ✓ All YAML files confirmed on disk.


## Cell 8 — Optional YAML validation with Cantera

This attempts to load each YAML with Cantera. If Cantera is not installed in the current environment, set `VALIDATE_YAML_WITH_CANTERA=False` in Cell 1 or install Cantera first.


In [8]:
# ── YAML validation ──────────────────────────────────────────────────────
print("=== Cantera YAML validation ===")
print("Loading each YAML with Cantera to verify structure before running equilibrium.")
print("This catches malformed coefficient blocks or unsupported thermo models early.")
print()

if VALIDATE_YAML_WITH_CANTERA:
    failed = []
    for path in single_manifest["yaml_path"]:
        abs_path = PATHS.root / path
        try:
            validate_yaml_with_cantera(abs_path, phase_name=PHASE_NAME)
            print(f"  ✓ {Path(path).name}")
        except Exception as e:
            print(f"  ✗ {Path(path).name}  →  {e}")
            failed.append(path)
    print()
    if failed:
        print(f"✗ {len(failed)} YAML(s) failed to load — check coefficient table and formula entries.")
    else:
        print(f"✓ All {len(single_manifest)} YAML files loaded successfully with Cantera.")
else:
    print("Skipped. Set VALIDATE_YAML_WITH_CANTERA=True in Cell 1 to run.")


=== Cantera YAML validation ===
Loading each YAML with Cantera to verify structure before running equilibrium.
This catches malformed coefficient blocks or unsupported thermo models early.

  ✓ Adenine.yaml
  ✓ Cytosine.yaml

✓ All 2 YAML files loaded successfully with Cantera.


## Cell 9 — Run Cantera equilibrium sweep

This runs all scenario-specific YAMLs across the selected diagnostic temperatures.

Outputs:

- `data/results/equilibrium_raw_long.csv`
- `data/results/equilibrium_raw_wide.csv`

The long table is the master table. The wide table is mostly for quick inspection.


In [9]:
RAW_LONG_CSV = PATHS.data_results / "equilibrium_raw_long.csv"
RAW_WIDE_CSV = PATHS.data_results / "equilibrium_raw_wide.csv"

if RUN_CANTERA_EQUILIBRIUM:
    print("=== Running Cantera equilibrium sweep ===")
    print(f"  {len(single_manifest)} YAML(s) × {len(EQUILIBRIUM_TEMPERATURES_C)} temperature(s) = "
          f"{len(single_manifest) * len(EQUILIBRIUM_TEMPERATURES_C)} runs")
    print(f"  Temperatures: {EQUILIBRIUM_TEMPERATURES_C} °C")
    print(f"  Solver: {CANTERA_SOLVER}   Pressure: {PRESSURE_PA:.0e} Pa")
    print()
    raw_long_df = run_equilibrium_manifest(
        manifest_df=single_manifest,
        scenarios=scenarios,
        temperatures_C=EQUILIBRIUM_TEMPERATURES_C,
        pressure_Pa=PRESSURE_PA,
        output_long_csv=RAW_LONG_CSV,
        phase_name=PHASE_NAME,
        solver=CANTERA_SOLVER,
        max_steps=CANTERA_MAX_STEPS,
    )
    raw_wide_df = make_raw_wide(raw_long_df, RAW_WIDE_CSV)
else:
    raw_long_df = pd.read_csv(RAW_LONG_CSV)
    raw_wide_df = pd.read_csv(RAW_WIDE_CSV)
    print("Loaded existing equilibrium results from disk (RUN_CANTERA_EQUILIBRIUM=False).")

# ── Run outcome summary ──────────────────────────────────────────────────
print()
print("=== Equilibrium run summary ===")
status_counts = raw_long_df.drop_duplicates(
    subset=["scenario", "yaml_file", "target_product", "T_C"]
)["solver_status"].value_counts()
for status, n in status_counts.items():
    icon = "✓" if status == "ok" else "✗"
    print(f"  {icon} {status}: {n} run(s)")

failed_runs = raw_long_df[raw_long_df["solver_status"] != "ok"][
    ["scenario", "target_product", "T_C", "solver_status", "error_message"]
].drop_duplicates(subset=["scenario", "target_product", "T_C"])
if not failed_runs.empty:
    print()
    print("  Failed runs:")
    display(failed_runs)
    print()
    print("  NOTE: Solver failures are usually caused by ill-conditioned initial")
    print("  compositions or temperature outside the NASA9 fit range. Check the")
    print("  YAML and coefficient ranges for the failing species.")
else:
    print()
    print("  ✓ All runs converged. Output rows:", len(raw_long_df))


=== Running Cantera equilibrium sweep ===
  2 YAML(s) × 1 temperature(s) = 2 runs
  Temperatures: [20.0] °C
  Solver: vcs   Pressure: 1e+05 Pa


=== Equilibrium run summary ===
  ✓ ok: 2 run(s)

  ✓ All runs converged. Output rows: 10


## Cell 10 — Convert mole fractions to moles using elemental balance

Cantera returns equilibrium mole fractions. For raw inspection and depletion diagnostics, this notebook reconstructs equilibrium moles using elemental conservation.

For each run, the code estimates total final moles from each conserved element and uses the median estimate. The column `element_balance_relative_spread` flags disagreement among element-wise estimates. Small values are good.

Output:

- `data/results/equilibrium_moles_long.csv`


In [10]:
MOLES_LONG_CSV = PATHS.data_results / "equilibrium_moles_long.csv"

moles_long_df = add_equilibrium_moles(raw_long_df, species_df)
moles_long_df.to_csv(MOLES_LONG_CSV, index=False)

# ── Mole balance diagnostics ─────────────────────────────────────────────
print("=== Equilibrium mole reconstruction ===")
print()
print("Cantera returns mole FRACTIONS (X_eq), not absolute moles.")
print("To get moles, we use elemental conservation:")
print("  For each element E:  N_total = (initial element-moles of E) / sum(X_i × atoms_E_i)")
print("We compute this for every element independently, then take the MEDIAN as the estimate.")
print()
print("The 'element_balance_relative_spread' column measures how much the per-element")
print("estimates of N_total disagree with each other (max−min / median).")
print()
print("Interpreting the spread:")
print("  ≈ 0     → all elements give identical N_total estimates (ideal)")
print("  small   → estimate is reliable")
print("  large   → inspect the run; in single-product mode, a product rich in a")
print("            minor element can make one elemental estimate diverge from the others.")
print()

spread_stats = moles_long_df.groupby(["target_product", "T_C"])["element_balance_relative_spread"].first()
spread_pivot = spread_stats.unstack("T_C").round(2)
print("element_balance_relative_spread by (product, temperature):")
display(spread_pivot)

max_spread = moles_long_df["element_balance_relative_spread"].max()
print()
if max_spread > 1.0:
    print(f"  ⚠  Max spread = {max_spread:.2f}. Inspect high-spread runs before interpreting absolute moles.")
else:
    print(f"  ✓ Max spread = {max_spread:.2f} — all element estimates are tightly consistent.")

print()
print(f"Saved: {MOLES_LONG_CSV}")


=== Equilibrium mole reconstruction ===

Cantera returns mole FRACTIONS (X_eq), not absolute moles.
To get moles, we use elemental conservation:
  For each element E:  N_total = (initial element-moles of E) / sum(X_i × atoms_E_i)
We compute this for every element independently, then take the MEDIAN as the estimate.

The 'element_balance_relative_spread' column measures how much the per-element
estimates of N_total disagree with each other (max−min / median).

Interpreting the spread:
  ≈ 0     → all elements give identical N_total estimates (ideal)
  small   → estimate is reliable
  large   → inspect the run; in single-product mode, a product rich in a
            minor element can make one elemental estimate diverge from the others.

element_balance_relative_spread by (product, temperature):


T_C,20.0
target_product,
Adenine(aq),0.0
Cytosine(aq),0.0



  ✓ Max spread = 0.00 — all element estimates are tightly consistent.

Saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/equilibrium_moles_long.csv


## Cell 11 — Raw equilibrium inspection table

This cell gives a table-format raw output for manual checking. It keeps the species-level view you are used to from earlier models, but includes reconstructed equilibrium moles and target-level accessibility calls.

Output:

- `data/results/equilibrium_inspection_table.csv`


In [11]:
INSPECTION_CSV = PATHS.data_results / "equilibrium_inspection_table.csv"

# A first-pass formation summary is needed only to annotate the raw inspection table.
inspection_formation_df = summarize_target_formation(
    moles_long_df=moles_long_df,
    x_threshold=FORMATION_X_THRESHOLD,
    n_threshold_mol=FORMATION_N_THRESHOLD_MOL,
    x_significant_threshold=SIGNIFICANT_X_THRESHOLD,
    output_csv=None,
)

inspection_df = make_equilibrium_inspection_table(
    moles_long_df=moles_long_df,
    formation_df=inspection_formation_df,
    scenarios=scenarios,
    output_csv=INSPECTION_CSV,
)

print("=== Raw equilibrium inspection table ===")
print()
print("This is a species-level table for debugging and sanity checks.")
print("Each row is one species inside one scenario × target-product × temperature run.")
print()
inspection_view = inspection_df[[c for c in [
    "scenario", "target_product", "T_C", "species", "X_initial", "X_eq",
    "initial_moles", "n_eq_mol", "formation_call", "solver_status"
] if c in inspection_df.columns]]
display(inspection_view)
print(f"Saved: {INSPECTION_CSV}")


=== Raw equilibrium inspection table ===

This is a species-level table for debugging and sanity checks.
Each row is one species inside one scenario × target-product × temperature run.



,scenario,target_product,T_C,species,X_initial,X_eq,initial_moles,n_eq_mol,formation_call,solver_status
0,validation,Adenine(aq),20.0,Adenine(aq),0.000000,3.752345e-03,0.000,4.000000e-03,significant,ok
1,validation,Adenine(aq),20.0,C2H2(aq),0.038817,3.939962e-02,0.042,4.200000e-02,significant,ok
2,validation,Adenine(aq),20.0,H2O(l),0.924214,9.380863e-01,1.000,1.000000e+00,significant,ok
3,validation,Adenine(aq),20.0,HCN(aq),0.018484,1.868258e-11,0.020,1.991564e-11,significant,ok
4,validation,Adenine(aq),20.0,NH3(aq),0.018484,1.876173e-02,0.020,2.000000e-02,significant,ok
5,validation,Cytosine(aq),20.0,C2H2(aq),0.038817,3.766478e-02,0.042,4.000000e-02,significant,ok
6,validation,Cytosine(aq),20.0,Cytosine(aq),0.000000,5.649718e-03,0.000,6.000000e-03,significant,ok
7,validation,Cytosine(aq),20.0,H2O(l),0.924214,9.359699e-01,1.000,9.940000e-01,significant,ok
8,validation,Cytosine(aq),20.0,HCN(aq),0.018484,2.306135e-16,0.020,2.449115e-16,significant,ok
9,validation,Cytosine(aq),20.0,NH3(aq),0.018484,2.071563e-02,0.020,2.200000e-02,significant,ok


Saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/equilibrium_inspection_table.csv


## Cell 12 — Target accessibility summary

This is the fast inspection cell for the main scientific question:

> Given the current starting inventory, do the target products appear at equilibrium?

A target is marked as accessible when the solver succeeds and `X_eq >= FORMATION_X_THRESHOLD`. This is a thermodynamic equilibrium-accessibility diagnostic, not a kinetic pathway claim.

Output:

- `data/results/target_formation_summary.csv`


In [12]:
TARGET_FORMATION_CSV = PATHS.data_results / "target_formation_summary.csv"

formation_df = summarize_target_formation(
    moles_long_df=moles_long_df,
    x_threshold=FORMATION_X_THRESHOLD,
    n_threshold_mol=FORMATION_N_THRESHOLD_MOL,
    x_significant_threshold=SIGNIFICANT_X_THRESHOLD,
    output_csv=TARGET_FORMATION_CSV,
)

# ── Formation diagnostics ─────────────────────────────────────────────────
print("=== Target accessibility summary ===")
print()
print("A target product is called accessible when ALL of these are true:")
print(f"  1. Solver converged (solver_status == 'ok')")
print(f"  2. X_eq >= {FORMATION_X_THRESHOLD:.0e}   (reporting threshold)")
print(f"     X_eq >= {SIGNIFICANT_X_THRESHOLD:.0e}  → 'significant'  |  below → 'trace'  (two-tier reporting)")
print(f"  3. n_eq_mol >= {FORMATION_N_THRESHOLD_MOL}  (equilibrium moles above minimum)")
print()
print("This is a thermodynamic ACCESSIBILITY diagnostic — it answers:")
print("  'Is this product thermodynamically stable in this mixture when included in the YAML?'")
print("It does NOT predict kinetics, reaction pathways, or synthesis yields.")
print()

formation_view = formation_df[[
    c for c in [
        "scenario", "target_product", "T_C", "X_eq", "log10_X_eq", "n_eq_mol",
        "formed_bool", "formation_call", "solver_status", "element_balance_relative_spread"
    ] if c in formation_df.columns
]]
display(formation_view)

# Summary counts
n_significant = (formation_df["formation_call"] == "significant").sum()
n_trace = (formation_df["formation_call"] == "trace").sum()
n_formed = formation_df["formed_bool"].sum()
n_total = len(formation_df)
n_failed = (formation_df["formation_call"] == "solver_failed").sum()
n_below = (formation_df["formation_call"] == "below_threshold").sum()
print()
print(f"  Significant (X_eq >= {SIGNIFICANT_X_THRESHOLD:.0e}) : {n_significant} / {n_total}")
print(f"  Trace       (X_eq >= {FORMATION_X_THRESHOLD:.0e}) : {n_trace} / {n_total}")
print(f"  Below threshold                       : {n_below} / {n_total}")
print(f"  Accessible total (sig + trace)        : {n_formed} / {n_total}")
print(f"  Solver failures                       : {n_failed}")
print()
if n_formed > 0:
    formed_products = formation_df[formation_df["formed_bool"]]["target_product"].unique()
    print(f"  Products above threshold at equilibrium: {sorted(formed_products)}")
else:
    print("  No target products are above threshold.")
    print("  Possible reasons: product is thermodynamically unfavourable at these")
    print("  conditions, or the starting inventory does not supply the right elements.")

print(f"\nSaved: {TARGET_FORMATION_CSV}")


=== Target accessibility summary ===

A target product is called accessible when ALL of these are true:
  1. Solver converged (solver_status == 'ok')
  2. X_eq >= 1e-12   (reporting threshold)
     X_eq >= 1e-06  → 'significant'  |  below → 'trace'  (two-tier reporting)
  3. n_eq_mol >= 0.0  (equilibrium moles above minimum)

This is a thermodynamic ACCESSIBILITY diagnostic — it answers:
  'Is this product thermodynamically stable in this mixture when included in the YAML?'
It does NOT predict kinetics, reaction pathways, or synthesis yields.



,scenario,target_product,T_C,X_eq,log10_X_eq,n_eq_mol,formed_bool,formation_call,solver_status,element_balance_relative_spread
0,validation,Adenine(aq),20.0,0.003752,-2.425697,0.004,True,significant,ok,4.165940e-16
1,validation,Cytosine(aq),20.0,0.005650,-2.247973,0.006,True,significant,ok,2.090815e-16



  Significant (X_eq >= 1e-06) : 2 / 2
  Trace       (X_eq >= 1e-12) : 0 / 2
  Below threshold                       : 0 / 2
  Accessible total (sig + trace)        : 2 / 2
  Solver failures                       : 0

  Products above threshold at equilibrium: ['Adenine(aq)', 'Cytosine(aq)']

Saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/target_formation_summary.csv


## Cell 13 — Reactant depletion/source diagnostic

This replaces the old limiting-reagent/yield branch.

For each starting species:

```text
depletion_fraction = (initial_moles - equilibrium_moles) / initial_moles
```

This answers a narrower and cleaner question:

> When a target product is allowed in the equilibrium calculation, which starting species changed the most?

This is **not** a pathway claim and **not** a formal limiting reagent analysis.

Outputs:

- `data/results/reactant_depletion_long.csv`
- `data/results/reactant_depletion_summary.csv`


In [13]:
DEPLETION_LONG_CSV = PATHS.data_results / "reactant_depletion_long.csv"
DEPLETION_SUMMARY_CSV = PATHS.data_results / "reactant_depletion_summary.csv"

# Starting species are those that appear in any scenario's initial inventory or extra_allowed_species.
starting_species = sorted({
    sp
    for cfg in scenarios["scenarios"].values()
    for sp in base_species_for_scenario(cfg)
})

# Target products should not be treated as starting inventory for depletion diagnostics.
starting_species = [sp for sp in starting_species if sp not in target_products]

depletion_long_df, depletion_summary_df = summarize_reactant_depletion(
    moles_long_df=moles_long_df,
    formation_df=formation_df,
    starting_species=starting_species,
    target_products=target_products,
    output_long_csv=DEPLETION_LONG_CSV,
    output_summary_csv=DEPLETION_SUMMARY_CSV,
)

# ── Depletion diagnostics ────────────────────────────────────────────────
print("=== Reactant depletion/source diagnostic ===")
print()
print("depletion_fraction = (initial_moles − equilibrium_moles) / initial_moles")
print()
print("  > 0  → species was consumed during equilibrium redistribution")
print("  ≈ 0  → species barely changed")
print("  < 0  → species was net-produced")
print()
print("This diagnostic reports the most depleted starting species and the largest")
print("absolute mole consumption. It does not identify a reaction pathway.")
print()

if depletion_long_df.empty:
    print("No depletion rows generated. Check starting inventory species and moles_long_df.")
else:
    dep_pivot = depletion_long_df.groupby(["species", "T_C"])["depletion_fraction"].mean().unstack("T_C").round(4)
    print("Mean depletion fraction by (starting species, temperature):")
    display(dep_pivot)

print()
print("=== Depletion summary per target run ===")
summary_view = depletion_summary_df[[c for c in [
    "scenario", "target_product", "T_C", "formation_call", "target_X_eq", "target_n_eq_mol",
    "most_depleted_species", "max_depletion_fraction", "most_consumed_species", "max_consumed_mol",
    "depletion_call", "solver_status"
] if c in depletion_summary_df.columns]]
display(summary_view)
print()
print(f"Long table saved   : {DEPLETION_LONG_CSV}")
print(f"Summary table saved: {DEPLETION_SUMMARY_CSV}")


=== Reactant depletion/source diagnostic ===

depletion_fraction = (initial_moles − equilibrium_moles) / initial_moles

  > 0  → species was consumed during equilibrium redistribution
  ≈ 0  → species barely changed
  < 0  → species was net-produced

This diagnostic reports the most depleted starting species and the largest
absolute mole consumption. It does not identify a reaction pathway.

Mean depletion fraction by (starting species, temperature):


T_C,20.0
species,
C2H2(aq),0.0238
H2O(l),0.0030
HCN(aq),1.0000
NH3(aq),-0.0500



=== Depletion summary per target run ===


,scenario,target_product,T_C,formation_call,target_X_eq,target_n_eq_mol,most_depleted_species,max_depletion_fraction,most_consumed_species,max_consumed_mol,depletion_call,solver_status
0,validation,Adenine(aq),20.0,significant,0.003752,0.004,HCN(aq),1.0,HCN(aq),0.02,dominant_depletion_detected,ok
1,validation,Cytosine(aq),20.0,significant,0.005650,0.006,HCN(aq),1.0,HCN(aq),0.02,dominant_depletion_detected,ok



Long table saved   : /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/reactant_depletion_long.csv
Summary table saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/reactant_depletion_summary.csv


## Cell 14 — Generate accessibility plots

This creates one combined accessibility bar chart per scenario and equilibrium temperature. The plot uses the v4 `plotting.py` function and the x-axis is controlled from Cell 1.

Output:

- `figures/equilibrium/*_combined_accessibility_barchart.pdf`
- matching PNG files when `PLOT_SAVE_PNG=True`


In [14]:
# Clear stale equilibrium figures so the folder matches this run.
for _ext in ("*.pdf", "*.png"):
    for _f in PATHS.figures_equilibrium.glob(_ext):
        _f.unlink()

print("=== Accessibility figures ===")
print()

for scenario_id in scenarios["scenarios"]:
    for T_C in EQUILIBRIUM_TEMPERATURES_C:
        combined_path = PATHS.figures_equilibrium / f"{scenario_id}_{T_C:g}C_combined_accessibility_barchart.pdf"

        plot_combined_accessibility_barchart(
            formation_df=formation_df,
            scenario=scenario_id,
            temperature_C=T_C,
            output_path=combined_path,
            formation_threshold=FORMATION_X_THRESHOLD,
            title=f"{scenario_id}  |  {T_C:g} °C",
            x_axis_max_mode=PLOT_X_AXIS_MAX_MODE,
            x_axis_max=PLOT_X_AXIS_MAX,
            x_axis_padding_factor=PLOT_X_AXIS_PADDING_FACTOR,
            figure_width=PLOT_FIGURE_WIDTH,
            save_png=PLOT_SAVE_PNG,
        )


=== Accessibility figures ===

  [combined_barchart] Saved → validation_20C_combined_accessibility_barchart.pdf


## Cell 15 — Final run summary

This cell writes a Markdown summary of what the notebook ran, including modeled conditions, starting inventory, equilibrium accessibility calls, depletion diagnostics, and saved output paths.

Output:

- `data/results/run_summary.md`


In [15]:
RUN_SUMMARY_MD = PATHS.data_results / "run_summary.md"

OUTPUT_PATHS = {
    "CHNOSZ Gibbs cache": CACHE_CSV,
    "Gibbs fitting table": GIBBS_WIDE_CSV,
    "NASA9 coefficients": COEFFS_CSV,
    "NASA9 fit diagnostics": FIT_DIAGNOSTICS_CSV,
    "Single-product YAML manifest": PATHS.models_single / "single_product_manifest.csv",
    "Raw equilibrium long table": RAW_LONG_CSV,
    "Raw equilibrium wide table": RAW_WIDE_CSV,
    "Equilibrium moles long table": MOLES_LONG_CSV,
    "Raw inspection table": INSPECTION_CSV,
    "Target accessibility summary": TARGET_FORMATION_CSV,
    "Reactant depletion long table": DEPLETION_LONG_CSV,
    "Reactant depletion summary": DEPLETION_SUMMARY_CSV,
    "Accessibility figures directory": PATHS.figures_equilibrium,
}

summary_path = write_run_summary(
    output_md=RUN_SUMMARY_MD,
    project_name=PROJECT_NAME,
    species_file=SPECIES_FILE,
    scenario_file=SCENARIO_FILE,
    scenarios=scenarios,
    target_products=target_products,
    equilibrium_temperatures_C=EQUILIBRIUM_TEMPERATURES_C,
    thermo_fit_temperatures_C=THERMO_FIT_TEMPERATURE_GRID_C,
    pressure_Pa=PRESSURE_PA,
    formation_x_threshold=FORMATION_X_THRESHOLD,
    significant_x_threshold=SIGNIFICANT_X_THRESHOLD,
    formation_n_threshold_mol=FORMATION_N_THRESHOLD_MOL,
    formation_df=formation_df,
    depletion_summary_df=depletion_summary_df,
    output_paths=OUTPUT_PATHS,
)

print("=== Final run summary ===")
print(f"Saved: {summary_path}")
print()
print("Important output files:")
for label, path in OUTPUT_PATHS.items():
    print(f"  {label:<34s} {path}")

print()
display(Markdown(summary_path.read_text(encoding="utf-8")))


=== Final run summary ===
Saved: /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/run_summary.md

Important output files:
  CHNOSZ Gibbs cache                 /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/raw/chnosz_gibbs_cache.csv
  Gibbs fitting table                /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/gibbs_for_fitting.csv
  NASA9 coefficients                 /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/nasa9_coefficients.csv
  NASA9 fit diagnostics              /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/nasa9_fit_diagnostics.csv
  Single-product YAML manifest       /Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/models/single_product/single_product_manifest.csv
  R

# Run summary — cantera_equilibrium_workflow_v4

## Modeled conditions

- Species file: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/inputs/species_example.csv`
- Scenario file: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/inputs/scenarios_example.yaml`
- Pressure: `100000 Pa` (`1 bar`)
- Equilibrium temperatures (°C): `[20.0]`
- Thermo fitting temperatures (°C): `[0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 120.0, 130.0, 140.0, 150.0, 160.0, 170.0, 180.0, 190.0, 200.0, 210.0, 220.0, 230.0, 240.0, 250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 310.0, 320.0, 330.0, 340.0, 350.0, 360.0, 370.0]`
- Formation threshold: `X_eq >= 1.000e-12`
- Significant threshold: `X_eq >= 1.000e-06`
- Minimum mole threshold: `n_eq_mol >= 0.000e+00`
- Target products modeled: `2`

## Starting inventory

### validation — Validation run: HCN + C2H2 + NH3 in water, 20 C

| Species | Initial moles |
|---|---:|
| `H2O(l)` | 1 |
| `HCN(aq)` | 0.02 |
| `C2H2(aq)` | 0.042 |
| `NH3(aq)` | 0.02 |

## Equilibrium accessibility summary

- Total target runs summarized: `2`
- significant: `2`

| scenario | target_product | T_C | X_eq | n_eq_mol | formation_call | solver_status |
|---|---|---|---|---|---|---|
| validation | Adenine(aq) | 20.0 | 3.752e-03 | 4.000e-03 | significant | ok |
| validation | Cytosine(aq) | 20.0 | 5.650e-03 | 6.000e-03 | significant | ok |

## Reactant depletion diagnostic

| scenario | target_product | T_C | formation_call | most_depleted_species | max_depletion_fraction | most_consumed_species | max_consumed_mol | depletion_call |
|---|---|---|---|---|---|---|---|---|
| validation | Adenine(aq) | 20.0 | significant | HCN(aq) | 1.000e+00 | HCN(aq) | 2.000e-02 | dominant_depletion_detected |
| validation | Cytosine(aq) | 20.0 | significant | HCN(aq) | 1.000e+00 | HCN(aq) | 2.000e-02 | dominant_depletion_detected |

## Saved outputs

- CHNOSZ Gibbs cache: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/raw/chnosz_gibbs_cache.csv`
- Gibbs fitting table: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/gibbs_for_fitting.csv`
- NASA9 coefficients: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/nasa9_coefficients.csv`
- NASA9 fit diagnostics: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/processed/nasa9_fit_diagnostics.csv`
- Single-product YAML manifest: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/models/single_product/single_product_manifest.csv`
- Raw equilibrium long table: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/equilibrium_raw_long.csv`
- Raw equilibrium wide table: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/equilibrium_raw_wide.csv`
- Equilibrium moles long table: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/equilibrium_moles_long.csv`
- Raw inspection table: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/equilibrium_inspection_table.csv`
- Target accessibility summary: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/target_formation_summary.csv`
- Reactant depletion long table: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/reactant_depletion_long.csv`
- Reactant depletion summary: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/data/results/reactant_depletion_summary.csv`
- Accessibility figures directory: `/Users/ishaanmadan/Project1_Thermochemical_Modeling/Kendra/cantera_equilibrium_workflow/figures/equilibrium`

## Interpretation note

These outputs report equilibrium accessibility for species explicitly included in each single-product YAML. They do not report kinetic rates, reaction pathways, or formal percent yields. The depletion diagnostic identifies which starting species changed most during equilibrium redistribution.
